# 11. 음이항 회귀 수렴 개선 및 강건성 비교

연속형 변수 표준화, Poisson 추정값 초기화, 여러 최적화 방법과 반복횟수를 사용하여 음이항 NB2 모형을 재추정한다. 수렴한 모형만 Poisson QMLE와 비교한다.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import *
setup_notebook()
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from IPython.display import display

data=load_model_data()
trips=load_trips().dropna(subset=['셀 ID'])
X_FINAL=['walk_20','walk_20_60','ic_access','competing_share','pop_20_50','buildings']
CONT=['ic_access','competing_share','pop_20_50','buildings']
HOURS={'오전(07~09시)':MORNING_HOURS,'종일':ALL_DAY_HOURS}

def build_pairs(hours):
    f=(trips[trips['시'].between(*hours)].pivot_table(index=['셀 ID','노선'],columns='연도',values='승차인원',aggfunc='sum',fill_value=0).reset_index().rename(columns={2024:'y2024',2025:'y2025'}))
    f['셀 ID']=f['셀 ID'].astype(int)
    f=f.merge(data[['셀 ID']+X_FINAL],on='셀 ID')
    return f[f.y2024>0].reset_index(drop=True)

def make_design(f, use_fe=True):
    x=f[X_FINAL].copy()
    for c in CONT:
        sd=x[c].std(ddof=0)
        x[c]=(x[c]-x[c].mean())/(sd if sd else 1)
    if use_fe:
        x=pd.concat([x,pd.get_dummies(f['노선'],prefix='노선',drop_first=True,dtype=float)],axis=1)
    return sm.add_constant(x,has_constant='add')
pairs={n:build_pairs(h) for n,h in HOURS.items()}
print({n:(len(f),f['셀 ID'].nunique(),f['노선'].nunique()) for n,f in pairs.items()})

{'오전(07~09시)': (186, 118, 24), '종일': (220, 132, 24)}


## 수렴 여부 점검 및 모형 사양 점검

In [2]:
def fit_nb_candidates(f, X, start):
    model=NegativeBinomial(f.y2025,X,offset=np.log(f.y2024),loglike_method='nb2')
    attempts=[]
    for method in ['bfgs','lbfgs','newton','nm']:
        try:
            r=model.fit(start_params=start,method=method,maxiter=5000,disp=False,cov_type='cluster',cov_kwds={'groups':pd.factorize(f['셀 ID'])[0],'use_correction':True})
            ret=getattr(r,'mle_retvals',{})
            attempts.append((method,r,ret.get('converged',False),ret.get('warnflag')))
            if ret.get('converged',False) and ret.get('warnflag',0)==0:
                return r,attempts
        except Exception as e:
            attempts.append((method,None,False,str(e)))
    return None,attempts

valid_nb={}
diagnostic_rows=[]
for period,f in pairs.items():
    for use_fe in [True,False]:
        X=make_design(f,use_fe=use_fe)
        pois=sm.GLM(f.y2025,X,family=sm.families.Poisson(),offset=np.log(f.y2024)).fit()
        start=np.r_[np.asarray(pois.params),0.5]
        nb_fit,attempts=fit_nb_candidates(f,X,start)
        diagnostic_rows.append({'시간대':period,'노선 FE':'포함' if use_fe else '제외','수렴':nb_fit is not None,'시도':[(a[0],a[2],a[3]) for a in attempts]})
        if nb_fit is not None and use_fe:
            valid_nb[period]=(nb_fit,X)
display(pd.DataFrame(diagnostic_rows))
print('converged=True이고 warnflag=0인 모형만 유효 음이항 결과로 사용한다.')

,시간대,노선 FE,수렴,시도
0,오전(07~09시),포함,True,"[(bfgs, True, 0)]"
1,오전(07~09시),제외,True,"[(bfgs, True, 0)]"
2,종일,포함,True,"[(bfgs, True, 0)]"
3,종일,제외,True,"[(bfgs, True, 0)]"


converged=True이고 warnflag=0인 모형만 유효 음이항 결과로 사용한다.


## 수렴한 음이항 모형과 Poisson QMLE 비교

In [3]:
rows=[]
for period,f in pairs.items():
    X=make_design(f,use_fe=True)
    pois=poisson_rate(f.y2025,f.y2024,X.drop(columns='const'),cluster=f['셀 ID'])
    for v in X_FINAL:
        rows.append({'시간대':period,'모형':'Poisson QMLE','변수':VARIABLE_LABELS.get(v,v),'계수':pois.params[v],'변화율(%)':(np.exp(pois.params[v])-1)*100,'p-value':pois.pvalues[v]})
    if period in valid_nb:
        nb_fit,_=valid_nb[period]
        for v in X_FINAL:
            rows.append({'시간대':period,'모형':'음이항 NB2 (수렴)','변수':VARIABLE_LABELS.get(v,v),'계수':nb_fit.params[v],'변화율(%)':(np.exp(nb_fit.params[v])-1)*100,'p-value':nb_fit.pvalues[v]})
comparison=pd.DataFrame(rows)
display(comparison.round({'계수':4,'변화율(%)':2,'p-value':4}))

key=['walk_20','walk_20_60','ic_access']
key_labels=[VARIABLE_LABELS.get(v,v) for v in key]
display(comparison[comparison['변수'].isin(key_labels)].round({'계수':4,'변화율(%)':2,'p-value':4}))

,시간대,모형,변수,계수,변화율(%),p-value
0,오전(07~09시),Poisson QMLE,보행 20분 이내,-0.5755,-43.76,0.0028
1,오전(07~09시),Poisson QMLE,보행 20~60분,-0.1887,-17.19,0.1632
2,오전(07~09시),Poisson QMLE,IC까지 도로거리(km),-0.2973,-25.72,0.0078
3,오전(07~09시),Poisson QMLE,GTX 경쟁 목적지 비율(%),-0.1167,-11.02,0.2932
4,오전(07~09시),Poisson QMLE,2024년 20~50대 인구,-0.0337,-3.31,0.6775
5,오전(07~09시),Poisson QMLE,2024년 건축물 수,-0.0390,-3.83,0.6671
6,오전(07~09시),음이항 NB2 (수렴),보행 20분 이내,0.1066,11.25,0.7237
7,오전(07~09시),음이항 NB2 (수렴),보행 20~60분,0.0753,7.82,0.7596
8,오전(07~09시),음이항 NB2 (수렴),IC까지 도로거리(km),-0.2543,-22.45,0.0093
9,오전(07~09시),음이항 NB2 (수렴),GTX 경쟁 목적지 비율(%),-0.0273,-2.70,0.7708


,시간대,모형,변수,계수,변화율(%),p-value
0,오전(07~09시),Poisson QMLE,보행 20분 이내,-0.5755,-43.76,0.0028
1,오전(07~09시),Poisson QMLE,보행 20~60분,-0.1887,-17.19,0.1632
2,오전(07~09시),Poisson QMLE,IC까지 도로거리(km),-0.2973,-25.72,0.0078
6,오전(07~09시),음이항 NB2 (수렴),보행 20분 이내,0.1066,11.25,0.7237
7,오전(07~09시),음이항 NB2 (수렴),보행 20~60분,0.0753,7.82,0.7596
8,오전(07~09시),음이항 NB2 (수렴),IC까지 도로거리(km),-0.2543,-22.45,0.0093
12,종일,Poisson QMLE,보행 20분 이내,-0.9257,-60.37,0.0000
13,종일,Poisson QMLE,보행 20~60분,-0.2870,-24.95,0.0142
14,종일,Poisson QMLE,IC까지 도로거리(km),-0.1537,-14.25,0.0168
18,종일,음이항 NB2 (수렴),보행 20분 이내,-0.8193,-55.93,0.0005
